# Validação de Fusão: openSMILE + Fingerprints Acústicos

Este notebook avalia empiricamente se as features de fingerprint acústico **complementam** as features openSMILE na predição de Valence e Arousal.

> **Nota metodológica:** o objetivo *não é provar previamente* que a fusão melhora, mas medir empiricamente se há ganho real e sob quais condições (target, tipo de fingerprint, modelo).

**Cenários avaliados:**
1. openSMILE isolado
2. Fingerprint Rank isolado
3. Fingerprint Band isolado
4. Todos os Fingerprints isolados
5. openSMILE + Fingerprint Rank
6. openSMILE + Fingerprint Band
7. openSMILE + Todos os Fingerprints
8. openSMILE + Raw Peaks (opcional)

**Validação:** GroupKFold(n_splits=5) agrupado por song_id — sem vazamento entre músicas.

## 1. Imports e configuração

In [1]:
import warnings
import re
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from scipy.stats import pearsonr, spearmanr
from sklearn.ensemble import (
    ExtraTreesRegressor,
    GradientBoostingRegressor,
    RandomForestRegressor,
)
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

try:
    from sklearn.ensemble import HistGradientBoostingRegressor
    HAS_HGBR = True
except ImportError:
    HAS_HGBR = False

warnings.filterwarnings('ignore')
print('HistGradientBoostingRegressor disponivel:', HAS_HGBR)

HistGradientBoostingRegressor disponivel: True


In [2]:
@dataclass
class Config:
    project_dir: Path = Path(r'C:\dev\python\TCC Ajustado')
    random_state: int = 42
    n_splits: int = 5
    n_estimators: int = 120
    n_jobs: int = -1
    selector_k: int = 60
    use_raw_peaks: bool = True
    export_html: bool = True
    show_figures: bool = True

    META_COLS: List[str] = field(default_factory=lambda: [
        'song_id', 'filename', 'block_id', 'block_start_sec', 'block_end_sec',
        'block_duration_sec', 'valence', 'arousal', 'quadrant', 'quadrant_label', 'method',
    ])
    TARGETS: List[str] = field(default_factory=lambda: ['valence', 'arousal'])

    def output_dir(self) -> Path:
        p = self.project_dir / 'Dados' / 'pycaret_fusion_validation_regression'
        p.mkdir(parents=True, exist_ok=True)
        return p

    def figures_dir(self) -> Path:
        p = self.output_dir() / 'figures'
        p.mkdir(parents=True, exist_ok=True)
        return p

    def data_dir(self) -> Path:
        return self.project_dir / 'Dados'

cfg = Config()
print('Output dir:', cfg.output_dir())
print('Figures dir:', cfg.figures_dir())

Output dir: C:\dev\python\TCC Ajustado\Dados\pycaret_fusion_validation_regression
Figures dir: C:\dev\python\TCC Ajustado\Dados\pycaret_fusion_validation_regression\figures


## 2. Carregamento dos dados

In [3]:
def numeric_feature_cols(df: pd.DataFrame, exclude: List[str]) -> List[str]:
    """Retorna colunas numéricas não-constantes que não são metadados."""
    out = []
    excl = set(exclude)
    for col in df.columns:
        if col in excl:
            continue
        s = pd.to_numeric(df[col], errors='coerce')
        if s.notna().sum() == 0 or s.nunique(dropna=True) <= 1:
            continue
        out.append(col)
    return out


def add_prefix(df: pd.DataFrame, prefix: str, exclude: List[str]) -> Tuple[pd.DataFrame, List[str]]:
    """Prefixa as colunas de feature; retorna df renomeado e lista de feature cols."""
    feat = [c for c in df.columns if c not in exclude]
    rename = {c: f'{prefix}{c}' for c in feat if not c.startswith(prefix)}
    df = df.rename(columns=rename)
    new_feat = [rename.get(c, c) for c in feat]
    return df, new_feat


def round_times(df: pd.DataFrame, decimals: int = 1) -> pd.DataFrame:
    """Arredonda start/end_sec para alinhar bases com pequenas diferenças."""
    for col in ['block_start_sec', 'block_end_sec']:
        if col in df.columns:
            df[col] = df[col].round(decimals)
    return df


def prefer_norm_magnitude(feat_list: List[str], df: pd.DataFrame) -> Tuple[List[str], List[str]]:
    """
    Remove features de magnitude em dB quando existe versão normalizada equivalente.

    Padrões substituídos (col_db → col_norm, se col_norm estiver no DataFrame):
      *_magnitude_db        → *_magnitude_norm_block    (fpband/fprank top-k por bloco)
      *_mag_*_db            → *_mag_*_norm_block        (fpband média/std por bloco)
      *_energy_db_<banda>   → *_energy_norm_<banda>     (fpband energia por banda)
      *_magnitude_db_<stat> → *_magnitude_norm_<stat>   (rawpeaks_band)
      *_magnitude_mean      → *_magnitude_mean_norm_block  (fprank média global sem sufixo _db)
    """
    df_cols = set(df.columns)
    keep: List[str] = []
    dropped: List[str] = []

    for col in feat_list:
        norm_col = None

        # padrão 1: sufixo _magnitude_db → _magnitude_norm_block
        if col.endswith('_magnitude_db'):
            norm_col = col[: -len('_magnitude_db')] + '_magnitude_norm_block'

        # padrão 2: contém _mag_ e termina em _db (ex: fp_low_mag_mean_db)
        elif col.endswith('_db') and '_mag_' in col and '_norm' not in col:
            norm_col = col[: -len('_db')] + '_norm_block'

        # padrão 3: _energy_db_<banda> → _energy_norm_<banda>
        elif 'energy_db_' in col:
            norm_col = col.replace('energy_db_', 'energy_norm_')

        # padrão 4: _magnitude_db_<stat> → _magnitude_norm_<stat>  (rawpeaks_band)
        elif '_magnitude_db_' in col:
            norm_col = col.replace('_magnitude_db_', '_magnitude_norm_')

        # padrão 5: terminado em _magnitude_mean sem _norm (ex: fprank__fp_magnitude_mean)
        elif '_norm_' not in col and col.endswith('_magnitude_mean'):
            norm_col = col + '_norm_block'

        if norm_col and norm_col in df_cols:
            dropped.append(col)
        else:
            keep.append(col)

    return keep, dropped

In [4]:
# ── openSMILE ──────────────────────────────────────────────────────────────
os_path = cfg.data_dir() / 'comparative_outputs' / 'common' / 'opensmile_blocks.parquet'
if not os_path.exists():
    raise FileNotFoundError(f'opensmile_blocks.parquet não encontrado: {os_path}')

os_df = pd.read_parquet(os_path)
os_df = round_times(os_df)

# as features openSMILE já vêm prefixadas com os_mean__; normalizamos para opensmile__
os_feat_raw = [c for c in os_df.columns if c.startswith('os_')]
os_rename = {c: c.replace('os_mean__', 'opensmile__').replace('os_', 'opensmile__') for c in os_feat_raw}
os_df = os_df.rename(columns=os_rename)
os_feat = list(os_rename.values())

print(f'openSMILE: {os_df.shape[0]} blocos | {len(os_feat)} features')
print('Keys:', os_df[['song_id','block_start_sec','block_end_sec']].dtypes.to_dict())

openSMILE: 6506 blocos | 520 features
Keys: {'song_id': dtype('int64'), 'block_start_sec': dtype('float64'), 'block_end_sec': dtype('float64')}


In [5]:
# ── Fingerprint Rank (bloco) ───────────────────────────────────────────────
fprank_path = cfg.data_dir() / 'fingerprint_rank' / 'fingerprint_rank.parquet'
fprank_df = pd.read_parquet(fprank_path)
fprank_df = round_times(fprank_df)
fprank_df, fprank_feat = add_prefix(fprank_df, 'fprank__', cfg.META_COLS)
fprank_feat = numeric_feature_cols(fprank_df, exclude=cfg.META_COLS)
print(f'FP Rank:  {fprank_df.shape[0]} blocos | {len(fprank_feat)} features')

# ── Fingerprint Band (bloco) ───────────────────────────────────────────────
fpband_path = cfg.data_dir() / 'fingerprint_band_rank' / 'fingerprint_band_rank.parquet'
fpband_df = pd.read_parquet(fpband_path)
fpband_df = round_times(fpband_df)
fpband_df, fpband_feat = add_prefix(fpband_df, 'fpband__', cfg.META_COLS)
fpband_feat = numeric_feature_cols(fpband_df, exclude=cfg.META_COLS)
print(f'FP Band:  {fpband_df.shape[0]} blocos | {len(fpband_feat)} features')

print(f'\nSongs em openSMILE: {os_df["song_id"].nunique()}')
print(f'Songs em FP Rank:   {fprank_df["song_id"].nunique()}')
print(f'Songs em FP Band:   {fpband_df["song_id"].nunique()}')

FP Rank:  6506 blocos | 207 features
FP Band:  6506 blocos | 302 features

Songs em openSMILE: 1802
Songs em FP Rank:   1802
Songs em FP Band:   1802


## 3. Agregação dos Raw Peaks (longo → bloco)

In [6]:
def aggregate_raw_peaks_rank(path: Path, prefix: str) -> Tuple[Optional[pd.DataFrame], List[str]]:
    """Agrega raw peaks (formato longo) para nível de bloco."""
    if not path.exists():
        return None, []
    raw = pd.read_parquet(path)
    raw = round_times(raw)
    keys = ['song_id', 'block_id', 'block_start_sec', 'block_end_sec']
    num_cols = ['frequency_hz', 'magnitude_db']
    if 'midi_note' in raw.columns:
        num_cols.append('midi_note')
    if 'octave' in raw.columns:
        num_cols.append('octave')
    if 'semitone' in raw.columns:
        num_cols.append('semitone')

    agg_dict = {}
    for col in num_cols:
        if col in raw.columns:
            agg_dict[f'{prefix}{col}_mean'] = (col, 'mean')
            agg_dict[f'{prefix}{col}_std']  = (col, 'std')
            agg_dict[f'{prefix}{col}_min']  = (col, 'min')
            agg_dict[f'{prefix}{col}_max']  = (col, 'max')
            agg_dict[f'{prefix}{col}_q25']  = (col, lambda x: x.quantile(0.25))
            agg_dict[f'{prefix}{col}_q75']  = (col, lambda x: x.quantile(0.75))
    agg_dict[f'{prefix}n_peaks'] = ('frequency_hz', 'count')

    # targets para preservar
    targets_in = [t for t in ['valence','arousal'] if t in raw.columns]
    for t in targets_in:
        agg_dict[t] = (t, 'first')

    grouped = raw.groupby(keys).agg(**{k: v for k, v in agg_dict.items()}).reset_index()
    feat_cols = [c for c in grouped.columns if c.startswith(prefix)]
    print(f'  {prefix}: {grouped.shape[0]} blocos | {len(feat_cols)} features (agregadas de {raw.shape[0]} peaks)')
    return grouped, feat_cols


def aggregate_raw_peaks_band(path: Path, prefix: str) -> Tuple[Optional[pd.DataFrame], List[str]]:
    """Agrega raw peaks por banda (formato longo) para nível de bloco."""
    if not path.exists():
        return None, []
    raw = pd.read_parquet(path)
    raw = round_times(raw)
    keys = ['song_id', 'block_id', 'block_start_sec', 'block_end_sec']
    num_cols = ['frequency_hz', 'magnitude_db', 'magnitude_norm']
    if 'midi_note' in raw.columns:
        num_cols.append('midi_note')

    # pivotar por banda
    pivot_frames = []
    bands = raw['band_name'].dropna().unique() if 'band_name' in raw.columns else []
    for band in bands:
        sub = raw[raw['band_name'] == band]
        agg_dict = {}
        for col in num_cols:
            if col in sub.columns:
                agg_dict[f'{prefix}{band}_{col}_mean'] = (col, 'mean')
                agg_dict[f'{prefix}{band}_{col}_std']  = (col, 'std')
                agg_dict[f'{prefix}{band}_{col}_max']  = (col, 'max')
        agg_dict[f'{prefix}{band}_n_peaks'] = ('frequency_hz', 'count')

        targets_in = [t for t in ['valence','arousal'] if t in sub.columns]
        for t in targets_in:
            agg_dict[t] = (t, 'first')

        g = sub.groupby(keys).agg(**{k: v for k, v in agg_dict.items()}).reset_index()
        pivot_frames.append(g)

    if not pivot_frames:
        return None, []

    merged = pivot_frames[0]
    target_cols = [t for t in ['valence','arousal'] if t in merged.columns]
    for pf in pivot_frames[1:]:
        drop_t = [t for t in target_cols if t in pf.columns]
        merged = merged.merge(pf.drop(columns=drop_t, errors='ignore'), on=keys, how='outer')

    feat_cols = [c for c in merged.columns if c.startswith(prefix)]
    print(f'  {prefix}: {merged.shape[0]} blocos | {len(feat_cols)} features (bandas: {list(bands)})')
    return merged, feat_cols

In [7]:
print('Agregando raw peaks...')
rawrank_df, rawrank_feat = aggregate_raw_peaks_rank(
    cfg.data_dir() / 'fingerprint_rank' / 'fingerprint_rank_raw_peaks.parquet',
    prefix='rawpeaks_rank__'
)
rawband_df, rawband_feat = aggregate_raw_peaks_band(
    cfg.data_dir() / 'fingerprint_band_rank' / 'fingerprint_band_rank_raw_peaks.parquet',
    prefix='rawpeaks_band__'
)

HAS_RAW = cfg.use_raw_peaks and (rawrank_df is not None or rawband_df is not None)
print(f'\nRaw peaks disponíveis: {HAS_RAW}')

Agregando raw peaks...
  rawpeaks_rank__: 6506 blocos | 31 features (agregadas de 195180 peaks)
  rawpeaks_band__: 6506 blocos | 52 features (bandas: ['low', 'mid', 'high', 'very_high'])

Raw peaks disponíveis: True


## 4. Montagem dos cenários de fusão

In [8]:
JOIN_KEYS = ['song_id', 'block_start_sec', 'block_end_sec']

# base comum: openSMILE (âncora do merge)
base_meta = ['song_id', 'filename', 'block_id', 'block_start_sec', 'block_end_sec',
             'block_duration_sec', 'valence', 'arousal']
base_meta = [c for c in base_meta if c in os_df.columns]

# dataset base = openSMILE
dataset = os_df[base_meta + os_feat].copy()

def left_join_source(base: pd.DataFrame, right: pd.DataFrame, feats: List[str], name: str) -> Tuple[pd.DataFrame, List[str]]:
    """Faz merge left do right no base pelas JOIN_KEYS; retorna df estendido e features adicionadas."""
    right_cols = [c for c in JOIN_KEYS if c in right.columns]
    right_sel = right[right_cols + feats].drop_duplicates(subset=right_cols)
    merged = base.merge(right_sel, on=right_cols, how='left')
    added = [f for f in feats if f in merged.columns]
    coverage = merged[added[0]].notna().mean() if added else 0.0
    print(f'  [{name}] blocos após merge: {merged.shape[0]} | coverage: {coverage:.1%}')
    return merged, added

# adicionar todas as fontes ao dataset principal
dataset, fprank_feat = left_join_source(dataset, fprank_df, fprank_feat, 'FP Rank')
dataset, fpband_feat = left_join_source(dataset, fpband_df, fpband_feat, 'FP Band')

if rawrank_df is not None:
    dataset, rawrank_feat = left_join_source(dataset, rawrank_df, rawrank_feat, 'Raw Rank')
if rawband_df is not None:
    dataset, rawband_feat = left_join_source(dataset, rawband_df, rawband_feat, 'Raw Band')

print(f'\nDataset total: {dataset.shape[0]} blocos x {dataset.shape[1]} colunas')
print(f'Songs: {dataset["song_id"].nunique()}')

  [FP Rank] blocos após merge: 6506 | coverage: 100.0%
  [FP Band] blocos após merge: 6506 | coverage: 100.0%
  [Raw Rank] blocos após merge: 6506 | coverage: 99.5%
  [Raw Band] blocos após merge: 6506 | coverage: 100.0%

Dataset total: 6506 blocos x 1120 colunas
Songs: 1802


In [9]:
# recalcular feature cols numéricas não-constantes no dataset integrado
all_meta = set(cfg.META_COLS + ['block_duration_sec', 'filename', 'block_id'])

def valid_feats(cols, df):
    out = []
    for c in cols:
        if c not in df.columns or c in all_meta:
            continue
        s = pd.to_numeric(df[c], errors='coerce')
        if s.notna().sum() > 0 and s.nunique(dropna=True) > 1:
            out.append(c)
    return out

os_feat_v      = valid_feats(os_feat, dataset)
fprank_feat_v  = valid_feats(fprank_feat, dataset)
fpband_feat_v  = valid_feats(fpband_feat, dataset)
rawrank_feat_v = valid_feats(rawrank_feat, dataset) if rawrank_feat else []
rawband_feat_v = valid_feats(rawband_feat, dataset) if rawband_feat else []

# ── Priorizar magnitude normalizada: remover versões em dB com par normalizado ──
fprank_feat_v,  fprank_dropped  = prefer_norm_magnitude(fprank_feat_v,  dataset)
fpband_feat_v,  fpband_dropped  = prefer_norm_magnitude(fpband_feat_v,  dataset)
rawrank_feat_v, rawrank_dropped = prefer_norm_magnitude(rawrank_feat_v, dataset)
rawband_feat_v, rawband_dropped = prefer_norm_magnitude(rawband_feat_v, dataset)

_all_dropped = fprank_dropped + fpband_dropped + rawrank_dropped + rawband_dropped
print(f'Features removidas (magnitude dB → normalizada disponível): {len(_all_dropped)}')
for _d in sorted(_all_dropped)[:30]:
    print(f'  - {_d}')
if len(_all_dropped) > 30:
    print(f'  ... e mais {len(_all_dropped) - 30}')
print()

all_fp  = list(dict.fromkeys(fprank_feat_v + fpband_feat_v))
all_raw = list(dict.fromkeys(rawrank_feat_v + rawband_feat_v))

print(f'openSMILE features válidas:     {len(os_feat_v)}')
print(f'FP Rank features válidas:       {len(fprank_feat_v)}')
print(f'FP Band features válidas:       {len(fpband_feat_v)}')
print(f'Raw Peaks features válidas:     {len(all_raw)}')

Features removidas (magnitude dB → normalizada disponível): 97
  - fpband__energy_db_high
  - fpband__energy_db_low
  - fpband__energy_db_mid
  - fpband__energy_db_very_high
  - fpband__fp_high_mag_mean_db
  - fpband__fp_high_mag_std_db
  - fpband__fp_high_top_10_magnitude_db
  - fpband__fp_high_top_1_magnitude_db
  - fpband__fp_high_top_2_magnitude_db
  - fpband__fp_high_top_3_magnitude_db
  - fpband__fp_high_top_4_magnitude_db
  - fpband__fp_high_top_5_magnitude_db
  - fpband__fp_high_top_6_magnitude_db
  - fpband__fp_high_top_7_magnitude_db
  - fpband__fp_high_top_8_magnitude_db
  - fpband__fp_high_top_9_magnitude_db
  - fpband__fp_low_mag_mean_db
  - fpband__fp_low_mag_std_db
  - fpband__fp_low_top_10_magnitude_db
  - fpband__fp_low_top_1_magnitude_db
  - fpband__fp_low_top_2_magnitude_db
  - fpband__fp_low_top_3_magnitude_db
  - fpband__fp_low_top_4_magnitude_db
  - fpband__fp_low_top_5_magnitude_db
  - fpband__fp_low_top_6_magnitude_db
  - fpband__fp_low_top_7_magnitude_db
  - fp

In [10]:
# definição dos cenários
SCENARIOS: Dict[str, List[str]] = {
    '1_opensmile_only':      os_feat_v,
    '2_fprank_only':         fprank_feat_v,
    '3_fpband_only':         fpband_feat_v,
    '4_allfingerprint_only': all_fp,
    '5_os_fprank':           os_feat_v + fprank_feat_v,
    '6_os_fpband':           os_feat_v + fpband_feat_v,
    '7_os_allfingerprint':   os_feat_v + all_fp,
}

if HAS_RAW and all_raw:
    SCENARIOS['8_os_rawpeaks'] = os_feat_v + all_raw

for name, feats in SCENARIOS.items():
    print(f'  {name}: {len(feats)} features')

# remover duplicatas dentro de cada cenário
SCENARIOS = {k: list(dict.fromkeys(v)) for k, v in SCENARIOS.items()}

  1_opensmile_only: 520 features
  2_fprank_only: 176 features
  3_fpband_only: 248 features
  4_allfingerprint_only: 424 features
  5_os_fprank: 696 features
  6_os_fpband: 768 features
  7_os_allfingerprint: 944 features
  8_os_rawpeaks: 586 features


## 5. Modelos e avaliação com GroupKFold

In [11]:
def get_models(cfg: Config) -> Dict[str, object]:
    models = {
        'Ridge':                    Ridge(alpha=1.0),
        'RandomForest':             RandomForestRegressor(n_estimators=cfg.n_estimators, random_state=cfg.random_state, n_jobs=cfg.n_jobs),
        'ExtraTrees':               ExtraTreesRegressor(n_estimators=cfg.n_estimators, random_state=cfg.random_state, n_jobs=cfg.n_jobs),
        'GradientBoosting':         GradientBoostingRegressor(n_estimators=cfg.n_estimators, random_state=cfg.random_state),
    }
    if HAS_HGBR:
        models['HistGradientBoosting'] = HistGradientBoostingRegressor(random_state=cfg.random_state)
    return models


def build_pipeline(model, scale: bool = True) -> Pipeline:
    steps = [('imputer', SimpleImputer(strategy='median'))]
    if scale:
        steps.append(('scaler', StandardScaler()))
    steps.append(('model', model))
    return Pipeline(steps)


def pearson_safe(y_true, y_pred) -> float:
    if np.std(y_pred) < 1e-9:
        return 0.0
    r, _ = pearsonr(y_true, y_pred)
    return float(r) if not np.isnan(r) else 0.0


def spearman_safe(y_true, y_pred) -> float:
    r, _ = spearmanr(y_true, y_pred)
    return float(r) if not np.isnan(r) else 0.0


def evaluate_fold(model, X_tr, y_tr, X_va, y_va) -> Dict[str, float]:
    model.fit(X_tr, y_tr)
    pred = model.predict(X_va)
    mae  = mean_absolute_error(y_va, pred)
    rmse = mean_squared_error(y_va, pred) ** 0.5
    r2   = r2_score(y_va, pred)
    pear = pearson_safe(y_va, pred)
    spear = spearman_safe(y_va, pred)
    return {'mae': mae, 'rmse': rmse, 'r2': r2, 'pearson': pear, 'spearman': spear}


def get_feature_importance(model_name: str, pipeline) -> Optional[np.ndarray]:
    est = pipeline.named_steps.get('model')
    if hasattr(est, 'feature_importances_'):
        return est.feature_importances_
    if hasattr(est, 'coef_'):
        return np.abs(est.coef_)
    return None

In [12]:
from tqdm.auto import tqdm

gkf = GroupKFold(n_splits=cfg.n_splits)
models = get_models(cfg)

fold_records = []
importance_records = []

targets_available = [t for t in cfg.TARGETS if t in dataset.columns]
groups = dataset['song_id'].values

total = len(SCENARIOS) * len(targets_available) * len(models)
pbar = tqdm(total=total, desc='Experimentos')

for scenario_name, feat_cols in SCENARIOS.items():
    # garantir que todas as features existem no dataset
    feat_cols = [f for f in feat_cols if f in dataset.columns]
    if not feat_cols:
        print(f'[WARN] {scenario_name}: sem features — pulando')
        continue

    X_all = dataset[feat_cols].apply(pd.to_numeric, errors='coerce')

    for target in targets_available:
        y_all = pd.to_numeric(dataset[target], errors='coerce')
        valid_mask = y_all.notna() & X_all.notna().any(axis=1)
        X_v = X_all[valid_mask].values
        y_v = y_all[valid_mask].values
        g_v = groups[valid_mask]

        for model_name, base_model in models.items():
            scale = model_name in ('Ridge',)
            fold_importances = []

            for fold_idx, (tr_idx, va_idx) in enumerate(gkf.split(X_v, y_v, g_v)):
                pipeline = build_pipeline(base_model.__class__(**base_model.get_params()), scale=scale)
                X_tr, X_va = X_v[tr_idx], X_v[va_idx]
                y_tr, y_va = y_v[tr_idx], y_v[va_idx]

                try:
                    metrics = evaluate_fold(pipeline, X_tr, y_tr, X_va, y_va)
                    status = 'ok'
                    imp = get_feature_importance(model_name, pipeline)
                    if imp is not None:
                        fold_importances.append(imp)
                except Exception as e:
                    metrics = {'mae': np.nan, 'rmse': np.nan, 'r2': np.nan, 'pearson': np.nan, 'spearman': np.nan}
                    status = f'error: {e}'

                fold_records.append({
                    'scenario': scenario_name,
                    'target': target,
                    'model': model_name,
                    'fold': fold_idx,
                    'n_features': len(feat_cols),
                    'n_train': len(tr_idx),
                    'n_val': len(va_idx),
                    'status': status,
                    **metrics,
                })

            # importância média entre folds
            if fold_importances:
                mean_imp = np.mean(fold_importances, axis=0)
                for feat, imp_val in zip(feat_cols, mean_imp):
                    importance_records.append({
                        'scenario': scenario_name,
                        'target': target,
                        'model': model_name,
                        'feature': feat,
                        'importance': imp_val,
                    })

            pbar.update(1)

pbar.close()
fold_df = pd.DataFrame(fold_records)
print(f'\nTotal de folds registrados: {len(fold_df)}')

Experimentos:   0%|          | 0/80 [00:00<?, ?it/s]

  File "c:\Users\felip\miniconda3\envs\pycaret311\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "c:\Users\felip\miniconda3\envs\pycaret311\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\felip\miniconda3\envs\pycaret311\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "c:\Users\felip\miniconda3\envs\pycaret311\Lib\subprocess.py", line 1538, in _execute_child
    hp, ht, pid, tid = _winapi.CreateProcess(executable, args,
                       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^



Total de folds registrados: 400


## 6. Agregação dos resultados

In [13]:
ok_df = fold_df[fold_df['status'] == 'ok'].copy()

agg_df = (
    ok_df.groupby(['scenario', 'target', 'model', 'n_features'])
    .agg(
        mae_mean=('mae', 'mean'),
        mae_std=('mae', 'std'),
        rmse_mean=('rmse', 'mean'),
        rmse_std=('rmse', 'std'),
        r2_mean=('r2', 'mean'),
        r2_std=('r2', 'std'),
        pearson_mean=('pearson', 'mean'),
        pearson_std=('pearson', 'std'),
        spearman_mean=('spearman', 'mean'),
        spearman_std=('spearman', 'std'),
        n_folds=('fold', 'count'),
    )
    .reset_index()
    .sort_values(['target', 'rmse_mean'])
)

print('Resultados agregados por cenário/target/modelo:')
display(agg_df.head(20))

Resultados agregados por cenário/target/modelo:


,scenario,target,model,n_features,mae_mean,mae_std,rmse_mean,rmse_std,r2_mean,r2_std,pearson_mean,pearson_std,spearman_mean,spearman_std,n_folds
62,7_os_allfingerprint,arousal,HistGradientBoosting,944,0.138853,0.002664,0.174654,0.002609,0.611898,0.029450,0.784083,0.016561,0.784593,0.015843,5
52,6_os_fpband,arousal,HistGradientBoosting,768,0.139956,0.003393,0.176285,0.003508,0.604832,0.027137,0.779754,0.015120,0.780029,0.013955,5
61,7_os_allfingerprint,arousal,GradientBoosting,944,0.140507,0.002713,0.176400,0.003486,0.604315,0.027248,0.779291,0.016045,0.779887,0.017498,5
42,5_os_fprank,arousal,HistGradientBoosting,696,0.140233,0.003771,0.177130,0.003600,0.601170,0.024716,0.776730,0.015721,0.777418,0.017072,5
51,6_os_fpband,arousal,GradientBoosting,768,0.141255,0.002937,0.177298,0.003907,0.600093,0.031178,0.776779,0.018480,0.777092,0.019937,5
60,7_os_allfingerprint,arousal,ExtraTrees,944,0.142182,0.002087,0.178622,0.002155,0.594191,0.027641,0.774727,0.019818,0.776130,0.018653,5
72,8_os_rawpeaks,arousal,HistGradientBoosting,586,0.141415,0.003832,0.178774,0.002909,0.593681,0.025616,0.772528,0.014013,0.773556,0.014252,5
53,6_os_fpband,arousal,RandomForest,768,0.142162,0.003091,0.179234,0.003606,0.591512,0.027377,0.771712,0.018805,0.773338,0.016806,5
50,6_os_fpband,arousal,ExtraTrees,768,0.142864,0.001521,0.179616,0.001544,0.589635,0.028083,0.771850,0.020188,0.773382,0.019360,5
63,7_os_allfingerprint,arousal,RandomForest,944,0.142849,0.001405,0.179645,0.002521,0.589320,0.031966,0.770204,0.021678,0.771366,0.019284,5


In [14]:
# melhor resultado por target e cenário (melhor modelo = menor RMSE médio)
best_by_scenario = (
    agg_df.sort_values('rmse_mean')
    .groupby(['target', 'scenario'])
    .first()
    .reset_index()
    .rename(columns={'model': 'best_model'})
)

# melhor cenário por target
best_by_target = (
    best_by_scenario.sort_values('rmse_mean')
    .groupby('target')
    .first()
    .reset_index()
)

print('Melhor cenário por target:')
display(best_by_target[['target', 'scenario', 'best_model', 'rmse_mean', 'mae_mean', 'r2_mean', 'pearson_mean']])

Melhor cenário por target:


,target,scenario,best_model,rmse_mean,mae_mean,r2_mean,pearson_mean
0,arousal,7_os_allfingerprint,HistGradientBoosting,0.174654,0.138853,0.611898,0.784083
1,valence,6_os_fpband,ExtraTrees,0.203371,0.162786,0.331549,0.593746


## 7. Comparação: cenários de fusão vs openSMILE isolado

In [15]:
def classify_gain(row) -> str:
    """Classifica o ganho em relação ao openSMILE isolado."""
    d_rmse = row.get('delta_rmse', 0.0)
    d_mae  = row.get('delta_mae', 0.0)
    d_r2   = row.get('delta_r2', 0.0)
    d_pear = row.get('delta_pearson', 0.0)

    # melhora = redução de erro (negativo) ou aumento de R²/Pearson (positivo)
    better_error = d_rmse < -0.005 and d_mae < -0.003
    better_expl  = d_r2 > 0.01 or d_pear > 0.01

    if better_error and better_expl:
        return 'melhora consistente'
    elif better_error:
        return 'melhora em erro'
    elif better_expl:
        return 'melhora em explicação'
    else:
        return 'sem melhora relevante'


# referência openSMILE por target e modelo
os_ref = agg_df[agg_df['scenario'] == '1_opensmile_only'][[
    'target', 'model', 'rmse_mean', 'mae_mean', 'r2_mean', 'pearson_mean'
]].rename(columns={
    'rmse_mean': 'os_rmse', 'mae_mean': 'os_mae',
    'r2_mean': 'os_r2', 'pearson_mean': 'os_pearson'
})

fusion_scenarios = [s for s in agg_df['scenario'].unique() if s != '1_opensmile_only']
fusion_df = agg_df[agg_df['scenario'].isin(fusion_scenarios)].copy()

delta_df = fusion_df.merge(os_ref, on=['target', 'model'], how='left')
delta_df['delta_rmse']    = delta_df['rmse_mean']    - delta_df['os_rmse']
delta_df['delta_mae']     = delta_df['mae_mean']     - delta_df['os_mae']
delta_df['delta_r2']      = delta_df['r2_mean']      - delta_df['os_r2']
delta_df['delta_pearson'] = delta_df['pearson_mean'] - delta_df['os_pearson']
delta_df['delta_rmse_pct'] = 100 * delta_df['delta_rmse'] / delta_df['os_rmse'].replace(0, np.nan)
delta_df['ganho'] = delta_df.apply(classify_gain, axis=1)

print('Comparação cenários de fusão vs openSMILE isolado:')
display(delta_df[['target','scenario','model','rmse_mean','os_rmse','delta_rmse','delta_rmse_pct','delta_r2','delta_pearson','ganho']].sort_values(['target','delta_rmse']))

Comparação cenários de fusão vs openSMILE isolado:


,target,scenario,model,rmse_mean,os_rmse,delta_rmse,delta_rmse_pct,delta_r2,delta_pearson,ganho
2,arousal,7_os_allfingerprint,GradientBoosting,0.176400,0.184762,-0.008361,-4.525386,0.038166,0.025042,melhora consistente
0,arousal,7_os_allfingerprint,HistGradientBoosting,0.174654,0.182175,-0.007521,-4.128467,0.034093,0.022255,melhora consistente
4,arousal,6_os_fpband,GradientBoosting,0.177298,0.184762,-0.007464,-4.039553,0.033944,0.022530,melhora consistente
5,arousal,7_os_allfingerprint,ExtraTrees,0.178622,0.184783,-0.006161,-3.334272,0.028121,0.018183,melhora consistente
1,arousal,6_os_fpband,HistGradientBoosting,0.176285,0.182175,-0.005890,-3.233113,0.027027,0.017925,melhora consistente
...,...,...,...,...,...,...,...,...,...,...
58,valence,2_fprank_only,HistGradientBoosting,0.228402,0.206640,0.021762,10.531159,-0.156035,-0.147737,sem melhora relevante
59,valence,3_fpband_only,RandomForest,0.236311,0.205387,0.030923,15.056145,-0.221385,-0.235946,sem melhora relevante
61,valence,3_fpband_only,ExtraTrees,0.237065,0.205108,0.031957,15.580758,-0.228579,-0.249467,sem melhora relevante
62,valence,3_fpband_only,GradientBoosting,0.239230,0.206185,0.033045,16.027115,-0.238364,-0.249762,sem melhora relevante


In [16]:
# resumo por cenário (melhor modelo de cada cenário)
scenario_summary = (
    delta_df.sort_values('rmse_mean')
    .groupby(['target','scenario'])
    .first()
    .reset_index()[['target','scenario','model','rmse_mean','os_rmse','delta_rmse','delta_rmse_pct','delta_r2','delta_pearson','ganho']]
    .rename(columns={'model':'best_model'})
)
print('Resumo por cenário (melhor modelo):')
display(scenario_summary)

Resumo por cenário (melhor modelo):


,target,scenario,best_model,rmse_mean,os_rmse,delta_rmse,delta_rmse_pct,delta_r2,delta_pearson,ganho
0,arousal,2_fprank_only,GradientBoosting,0.198024,0.184762,0.013263,7.178285,-0.065278,-0.043845,sem melhora relevante
1,arousal,3_fpband_only,GradientBoosting,0.192591,0.184762,0.007829,4.237516,-0.038170,-0.024482,sem melhora relevante
2,arousal,4_allfingerprint_only,HistGradientBoosting,0.189367,0.182175,0.007192,3.947950,-0.034203,-0.022209,sem melhora relevante
3,arousal,5_os_fprank,HistGradientBoosting,0.177130,0.182175,-0.005045,-2.769302,0.023365,0.014901,melhora consistente
4,arousal,6_os_fpband,HistGradientBoosting,0.176285,0.182175,-0.005890,-3.233113,0.027027,0.017925,melhora consistente
5,arousal,7_os_allfingerprint,HistGradientBoosting,0.174654,0.182175,-0.007521,-4.128467,0.034093,0.022255,melhora consistente
6,arousal,8_os_rawpeaks,HistGradientBoosting,0.178774,0.182175,-0.003401,-1.866988,0.015876,0.010699,melhora em explicação
7,valence,2_fprank_only,ExtraTrees,0.223477,0.205108,0.018369,8.955581,-0.130471,-0.127795,sem melhora relevante
8,valence,3_fpband_only,RandomForest,0.236311,0.205387,0.030923,15.056145,-0.221385,-0.235946,sem melhora relevante
9,valence,4_allfingerprint_only,RandomForest,0.217061,0.205387,0.011674,5.683669,-0.082945,-0.078110,sem melhora relevante


## 8. Importância das features por origem

In [17]:
importance_df = pd.DataFrame(importance_records)

def feature_origin(feat: str) -> str:
    if feat.startswith('opensmile__'):
        return 'openSMILE'
    if feat.startswith('fprank__'):
        return 'Fingerprint Rank'
    if feat.startswith('fpband__'):
        return 'Fingerprint Band'
    if feat.startswith('rawpeaks_rank__'):
        return 'Raw Peaks Rank'
    if feat.startswith('rawpeaks_band__'):
        return 'Raw Peaks Band'
    return 'outros'

if not importance_df.empty:
    importance_df['origin'] = importance_df['feature'].map(feature_origin)

    # importância por origem
    imp_by_origin = (
        importance_df.groupby(['scenario','target','model','origin'])['importance']
        .sum()
        .reset_index()
    )

    # normalizar por cenário/target/modelo
    total_imp = imp_by_origin.groupby(['scenario','target','model'])['importance'].transform('sum')
    imp_by_origin['importance_pct'] = 100 * imp_by_origin['importance'] / total_imp.replace(0, np.nan)

    print('Importância por origem (cenários de fusão):')
    fusion_imp = imp_by_origin[imp_by_origin['scenario'].isin(fusion_scenarios)]
    display(fusion_imp.sort_values(['target','scenario','importance_pct'], ascending=[True,True,False]).head(40))
else:
    print('[INFO] Nenhuma importância de feature registrada.')
    imp_by_origin = pd.DataFrame()

Importância por origem (cenários de fusão):


,scenario,target,model,origin,importance,importance_pct
8,2_fprank_only,arousal,ExtraTrees,Fingerprint Rank,1.000000,100.000000
9,2_fprank_only,arousal,GradientBoosting,Fingerprint Rank,1.000000,100.000000
10,2_fprank_only,arousal,RandomForest,Fingerprint Rank,1.000000,100.000000
11,2_fprank_only,arousal,Ridge,Fingerprint Rank,6.691057,100.000000
16,3_fpband_only,arousal,ExtraTrees,Fingerprint Band,1.000000,100.000000
17,3_fpband_only,arousal,GradientBoosting,Fingerprint Band,1.000000,100.000000
18,3_fpband_only,arousal,RandomForest,Fingerprint Band,1.000000,100.000000
19,3_fpband_only,arousal,Ridge,Fingerprint Band,9.014358,100.000000
26,4_allfingerprint_only,arousal,GradientBoosting,Fingerprint Band,0.795059,79.505949
28,4_allfingerprint_only,arousal,RandomForest,Fingerprint Band,0.736373,73.637322


In [18]:
# top 20 features por target (cenário 7: openSMILE + todos fingerprints)
from IPython.display import display, Markdown

for target in targets_available:
    sub = importance_df[
        (importance_df['target'] == target) &
        (importance_df['scenario'] == '7_os_allfingerprint') &
        (importance_df['model'].isin(['ExtraTrees','RandomForest']))
    ]
    if sub.empty:
        sub = importance_df[
            (importance_df['target'] == target) &
            (importance_df['scenario'] == '7_os_allfingerprint')
        ]
    if sub.empty:
        continue
    top20 = (
        sub.groupby('feature')['importance']
        .mean()
        .reset_index()
        .sort_values('importance', ascending=False)
        .head(20)
    )
    top20['origin'] = top20['feature'].map(feature_origin)
    display(Markdown(f'### Top 20 features — {target} (cenário 7)'))
    display(top20)

### Top 20 features — valence (cenário 7)

,feature,importance,origin
538,opensmile__audspec_lengthL1norm_sma_de_stddev,0.066608,openSMILE
550,opensmile__logHNR_sma_de_stddev,0.023935,openSMILE
630,opensmile__pcm_fftMag_spectralEntropy_sma_de_s...,0.021226,openSMILE
265,fprank__fp_peak_gap_std,0.017413,Fingerprint Rank
3,fpband__energy_norm_very_high,0.014815,Fingerprint Band
426,opensmile__F0final_sma_de_stddev,0.009516,openSMILE
253,fprank__fp_magnitude_mean_norm_block,0.008887,Fingerprint Rank
632,opensmile__pcm_fftMag_spectralFlux_sma_amean,0.008226,openSMILE
72,fpband__fp_low_top_10_magnitude_norm_block,0.008116,Fingerprint Band
247,fpband__score_very_high,0.007470,Fingerprint Band


### Top 20 features — arousal (cenário 7)

,feature,importance,origin
0,fpband__energy_norm_high,0.238156,Fingerprint Band
253,fprank__fp_magnitude_mean_norm_block,0.073779,Fingerprint Rank
550,opensmile__logHNR_sma_de_stddev,0.027823,openSMILE
574,opensmile__pcm_fftMag_mfcc_sma[1]_amean,0.026309,openSMILE
551,opensmile__logHNR_sma_stddev,0.017765,openSMILE
3,fpband__energy_norm_very_high,0.012831,Fingerprint Band
538,opensmile__audspec_lengthL1norm_sma_de_stddev,0.010055,openSMILE
536,opensmile__audspec_lengthL1norm_sma_amean,0.009771,openSMILE
630,opensmile__pcm_fftMag_spectralEntropy_sma_de_s...,0.008192,openSMILE
13,fpband__fp_high_top_10_magnitude_norm_block,0.007679,Fingerprint Band


## 9. Gráficos

In [19]:
SCENARIO_LABELS = {
    '1_opensmile_only':      '1. openSMILE',
    '2_fprank_only':         '2. FP Rank',
    '3_fpband_only':         '3. FP Band',
    '4_allfingerprint_only': '4. All FP',
    '5_os_fprank':           '5. OS+FPRank',
    '6_os_fpband':           '6. OS+FPBand',
    '7_os_allfingerprint':   '7. OS+AllFP',
    '8_os_rawpeaks':         '8. OS+RawPks',
}

SCENARIO_TYPE = {
    '1_opensmile_only':      'openSMILE',
    '2_fprank_only':         'Apenas FP',
    '3_fpband_only':         'Apenas FP',
    '4_allfingerprint_only': 'Apenas FP',
    '5_os_fprank':           'Fusao OS+FP',
    '6_os_fpband':           'Fusao OS+FP',
    '7_os_allfingerprint':   'Fusao OS+FP',
    '8_os_rawpeaks':         'Fusao OS+FP',
}

ORIGIN_COLORS = {
    'openSMILE':         '#2196F3',
    'Fingerprint Rank':  '#FF9800',
    'Fingerprint Band':  '#4CAF50',
    'Raw Peaks Rank':    '#9C27B0',
    'Raw Peaks Band':    '#E91E63',
    'outros':            '#9E9E9E',
}

SCENARIO_TYPE_COLORS = {
    'openSMILE':    '#2196F3',
    'Apenas FP':    '#FF9800',
    'Fusao OS+FP':  '#4CAF50',
}

PLOTLY_TEMPLATE = 'plotly_white'


def scenario_label(s: str) -> str:
    return SCENARIO_LABELS.get(s, s)


def scenario_type(s: str) -> str:
    return SCENARIO_TYPE.get(s, 'outros')


def save_plot(fig, name: str, cfg) -> None:
    if cfg.export_html:
        path = cfg.figures_dir() / f'{name}.html'
        fig.write_html(str(path), include_plotlyjs='cdn')
        print(f'Salvo: {path.name}')
    if cfg.show_figures:
        fig.show()

In [20]:
# Grafico 1: RMSE por cenario e target
plot_rmse = best_by_scenario.copy()
plot_rmse['label'] = plot_rmse['scenario'].map(scenario_label)
plot_rmse['tipo'] = plot_rmse['scenario'].map(scenario_type)

label_order_rmse = (
    plot_rmse.groupby('label')['rmse_mean'].mean()
    .sort_values(ascending=False)
    .index.tolist()
)

fig_rmse = px.bar(
    plot_rmse.sort_values('scenario'),
    x='rmse_mean',
    y='label',
    color='tipo',
    facet_col='target',
    orientation='h',
    error_x='rmse_std',
    text=plot_rmse.sort_values('scenario')['rmse_mean'].map('{:.4f}'.format),
    hover_data={
        'best_model': True,
        'n_features': True,
        'mae_mean': ':.4f',
        'r2_mean': ':.3f',
        'pearson_mean': ':.3f',
        'rmse_std': ':.4f',
        'tipo': False,
    },
    color_discrete_map=SCENARIO_TYPE_COLORS,
    category_orders={'label': label_order_rmse},
    title='RMSE por Cenario — melhor modelo (GroupKFold n=5 por song_id)',
    labels={'rmse_mean': 'RMSE medio', 'label': '', 'tipo': 'Tipo de cenario'},
)
fig_rmse.update_traces(textposition='outside')
fig_rmse.update_layout(
    template=PLOTLY_TEMPLATE,
    height=460,
    legend_title='Tipo de Cenario',
    margin=dict(l=10, r=60, t=60, b=10),
)
fig_rmse.for_each_annotation(lambda a: a.update(text=a.text.split('=')[-1].capitalize()))
fig_rmse.update_xaxes(matches=None)

save_plot(fig_rmse, 'fig_rmse_por_cenario', cfg)

Salvo: fig_rmse_por_cenario.html


In [21]:
# Grafico 2: delta RMSE vs openSMILE isolado
if not scenario_summary.empty:
    plot_delta = scenario_summary.copy()
    plot_delta['label'] = plot_delta['scenario'].map(scenario_label)
    plot_delta['cor'] = plot_delta['delta_rmse'].apply(
        lambda d: 'Melhorou' if d < 0 else 'Piorou'
    )

    label_order_delta = (
        plot_delta.groupby('label')['delta_rmse'].mean()
        .sort_values(ascending=True)
        .index.tolist()
    )

    fig_delta = px.bar(
        plot_delta.sort_values('scenario'),
        x='delta_rmse',
        y='label',
        color='cor',
        facet_col='target',
        orientation='h',
        text=plot_delta.sort_values('scenario')['delta_rmse'].map('{:+.4f}'.format),
        hover_data={
            'best_model': True,
            'rmse_mean': ':.4f',
            'os_rmse': ':.4f',
            'delta_rmse_pct': ':.2f',
            'delta_r2': ':.4f',
            'delta_pearson': ':.4f',
            'ganho': True,
            'cor': False,
        },
        color_discrete_map={'Melhorou': '#4CAF50', 'Piorou': '#F44336'},
        category_orders={'label': label_order_delta},
        title='Delta RMSE vs openSMILE isolado — valores negativos indicam melhora',
        labels={
            'delta_rmse': 'delta RMSE (fusao - openSMILE)',
            'label': '',
            'cor': 'Impacto',
        },
    )
    fig_delta.update_traces(textposition='outside')
    fig_delta.add_vline(x=0, line_dash='dash', line_color='black', line_width=1.5)
    fig_delta.update_layout(
        template=PLOTLY_TEMPLATE,
        height=460,
        legend_title='Impacto',
        margin=dict(l=10, r=60, t=60, b=10),
    )
    fig_delta.for_each_annotation(lambda a: a.update(text=a.text.split('=')[-1].capitalize()))
    fig_delta.update_xaxes(matches=None)

    save_plot(fig_delta, 'fig_delta_rmse_vs_opensmile', cfg)

Salvo: fig_delta_rmse_vs_opensmile.html


In [22]:
# Grafico 3: R2 por cenario
plot_r2 = best_by_scenario.copy()
plot_r2['label'] = plot_r2['scenario'].map(scenario_label)
plot_r2['tipo'] = plot_r2['scenario'].map(scenario_type)

label_order_r2 = (
    plot_r2.groupby('label')['r2_mean'].mean()
    .sort_values(ascending=True)
    .index.tolist()
)

fig_r2 = px.bar(
    plot_r2.sort_values('scenario'),
    x='r2_mean',
    y='label',
    color='tipo',
    facet_col='target',
    orientation='h',
    error_x='r2_std',
    text=plot_r2.sort_values('scenario')['r2_mean'].map('{:.3f}'.format),
    hover_data={
        'best_model': True,
        'n_features': True,
        'pearson_mean': ':.3f',
        'r2_std': ':.4f',
        'tipo': False,
    },
    color_discrete_map=SCENARIO_TYPE_COLORS,
    category_orders={'label': label_order_r2},
    title='R2 por Cenario — melhor modelo (GroupKFold n=5 por song_id)',
    labels={'r2_mean': 'R2 medio', 'label': '', 'tipo': 'Tipo de cenario'},
)
fig_r2.update_traces(textposition='outside')
fig_r2.add_vline(x=0, line_dash='dash', line_color='black', line_width=1.5)
fig_r2.update_layout(
    template=PLOTLY_TEMPLATE,
    height=460,
    legend_title='Tipo de Cenario',
    margin=dict(l=10, r=60, t=60, b=10),
)
fig_r2.for_each_annotation(lambda a: a.update(text=a.text.split('=')[-1].capitalize()))
fig_r2.update_xaxes(matches=None)

save_plot(fig_r2, 'fig_r2_por_cenario', cfg)

Salvo: fig_r2_por_cenario.html


In [23]:
# Grafico 4: importancia por origem (cenario 7 — OS+AllFP)
if not imp_by_origin.empty:
    fusion_imp_plot = imp_by_origin[
        imp_by_origin['scenario'] == '7_os_allfingerprint'
    ].copy()

    if not fusion_imp_plot.empty:
        grp_imp = (
            fusion_imp_plot
            .groupby(['target', 'origin'])['importance_pct']
            .mean()
            .reset_index()
            .sort_values('importance_pct', ascending=False)
        )

        fig_imp = px.bar(
            grp_imp,
            x='origin',
            y='importance_pct',
            color='origin',
            facet_col='target',
            text=grp_imp['importance_pct'].map('{:.1f}%'.format),
            hover_data={'importance_pct': ':.2f'},
            color_discrete_map=ORIGIN_COLORS,
            title='Importancia por Origem — Cenario OS+AllFP (ExtraTrees + RF + GradientBoosting)',
            labels={
                'importance_pct': 'Importancia relativa (%)',
                'origin': 'Origem',
            },
        )
        fig_imp.update_traces(textposition='outside', showlegend=False)
        fig_imp.update_layout(
            template=PLOTLY_TEMPLATE,
            height=450,
            margin=dict(l=10, r=10, t=60, b=10),
        )
        fig_imp.for_each_annotation(lambda a: a.update(text=a.text.split('=')[-1].capitalize()))
        fig_imp.update_yaxes(matches=None)

        save_plot(fig_imp, 'fig_importancia_por_origem', cfg)

Salvo: fig_importancia_por_origem.html


In [24]:
# Graficos 5-6: top 20 features por target (cenario 7 — OS+AllFP)
for target in targets_available:
    sub = importance_df[
        (importance_df['target'] == target) &
        (importance_df['scenario'] == '7_os_allfingerprint')
    ]
    if sub.empty:
        continue

    top20 = (
        sub.groupby('feature')['importance']
        .mean()
        .sort_values(ascending=False)
        .head(20)
        .reset_index()
    )
    top20['origin'] = top20['feature'].map(feature_origin)
    top20['short'] = (
        top20['feature']
        .str.replace('opensmile__', 'os:', regex=False)
        .str.replace('fprank__', 'fpr:', regex=False)
        .str.replace('fpband__', 'fpb:', regex=False)
        .str.replace('rawpeaks_rank__', 'rpr:', regex=False)
        .str.replace('rawpeaks_band__', 'rpb:', regex=False)
        .str[:60]
    )

    fig_top = px.bar(
        top20.sort_values('importance', ascending=True),
        x='importance',
        y='short',
        color='origin',
        orientation='h',
        text=top20.sort_values('importance', ascending=True)['importance'].map('{:.4f}'.format),
        hover_data={
            'feature': True,
            'importance': ':.5f',
            'origin': True,
            'short': False,
        },
        color_discrete_map=ORIGIN_COLORS,
        title=f'Top 20 Features — {target.capitalize()} (cenario OS+AllFP)',
        labels={
            'importance': 'Importancia media',
            'short': '',
            'origin': 'Origem',
        },
    )
    fig_top.update_traces(textposition='outside')
    fig_top.update_layout(
        template=PLOTLY_TEMPLATE,
        height=620,
        legend_title='Origem',
        margin=dict(l=10, r=80, t=60, b=10),
    )

    save_plot(fig_top, f'fig_top20_features_{target}', cfg)

Salvo: fig_top20_features_valence.html


Salvo: fig_top20_features_arousal.html


In [25]:
# Grafico 7: visao geral multi-metrica por cenario
# Compara RMSE, MAE, R2 e Pearson em paineis facetados por target e metrica

metrics_cfg = [
    ('rmse_mean',    'RMSE medio', 'rmse_std'),
    ('mae_mean',     'MAE medio',  'mae_std'),
    ('r2_mean',      'R2 medio',   'r2_std'),
    ('pearson_mean', 'Pearson r',  'pearson_std'),
]

multi_frames = []
for metric_col, metric_label, std_col in metrics_cfg:
    if metric_col not in best_by_scenario.columns:
        continue
    sub = best_by_scenario[['scenario', 'target', 'best_model', 'n_features',
                             metric_col, std_col]].copy()
    sub = sub.rename(columns={metric_col: 'valor', std_col: 'valor_std'})
    sub['metrica'] = metric_label
    sub['label'] = sub['scenario'].map(scenario_label)
    sub['tipo'] = sub['scenario'].map(scenario_type)
    multi_frames.append(sub)

if multi_frames:
    multi_df = pd.concat(multi_frames, ignore_index=True)

    fig_multi = px.bar(
        multi_df.sort_values('scenario'),
        x='label',
        y='valor',
        error_y='valor_std',
        color='tipo',
        facet_col='metrica',
        facet_row='target',
        barmode='group',
        text=multi_df.sort_values('scenario')['valor'].map('{:.3f}'.format),
        hover_data={
            'best_model': True,
            'n_features': True,
            'tipo': False,
            'valor_std': ':.4f',
        },
        color_discrete_map=SCENARIO_TYPE_COLORS,
        title='Visao Geral Multi-Metrica por Cenario (melhor modelo de cada cenario)',
        labels={'valor': 'Valor', 'label': 'Cenario', 'tipo': 'Tipo'},
    )
    fig_multi.update_traces(
        texttemplate='%{text}',
        textposition='outside',
        textfont_size=8,
    )
    fig_multi.update_layout(
        template=PLOTLY_TEMPLATE,
        height=750,
        legend_title='Tipo de Cenario',
        xaxis_tickangle=-40,
        margin=dict(l=10, r=10, t=80, b=60),
    )
    fig_multi.for_each_annotation(lambda a: a.update(text=a.text.split('=')[-1]))
    fig_multi.update_yaxes(matches=None)

    save_plot(fig_multi, 'fig_multimetrica_cenarios', cfg)

Salvo: fig_multimetrica_cenarios.html


## 10. Exportação dos resultados

In [26]:
out = cfg.output_dir()

fold_df.to_csv(out / 'fusion_fold_results.csv', index=False)
agg_df.to_csv(out / 'fusion_all_model_results.csv', index=False)
best_by_target.to_csv(out / 'fusion_best_by_target.csv', index=False)

if not delta_df.empty:
    delta_df.to_csv(out / 'fusion_delta_vs_opensmile.csv', index=False)

if not scenario_summary.empty:
    scenario_summary.to_csv(out / 'fusion_scenario_summary.csv', index=False)

if not importance_df.empty:
    importance_df.to_csv(out / 'fusion_feature_importance.csv', index=False)

if not imp_by_origin.empty:
    imp_by_origin.to_csv(out / 'fusion_feature_importance_by_origin.csv', index=False)

print('Arquivos exportados para:', out)
for f in sorted(out.glob('*.csv')):
    print(f'  {f.name}')

print('Figuras HTML salvas em:', cfg.figures_dir())
for f in sorted(cfg.figures_dir().glob('*.html')):
    print(f'  {f.name}')

Arquivos exportados para: C:\dev\python\TCC Ajustado\Dados\pycaret_fusion_validation_regression
  fusion_all_model_results.csv
  fusion_best_by_target.csv
  fusion_delta_vs_opensmile.csv
  fusion_feature_importance.csv
  fusion_feature_importance_by_origin.csv
  fusion_fold_results.csv
  fusion_scenario_summary.csv
Figuras HTML salvas em: C:\dev\python\TCC Ajustado\Dados\pycaret_fusion_validation_regression\figures
  fig_delta_rmse_vs_opensmile.html
  fig_importancia_por_origem.html
  fig_multimetrica_cenarios.html
  fig_r2_por_cenario.html
  fig_rmse_por_cenario.html
  fig_top20_features_arousal.html
  fig_top20_features_valence.html


## 11. Síntese final

In [27]:
from IPython.display import Markdown, display

def sintetizar(delta_df, scenario_summary, imp_by_origin, targets_available, cfg):
    linhas = [
        '# Síntese: Fusão openSMILE + Fingerprints Acústicos',
        '',
        '> Este resumo foi gerado automaticamente a partir dos resultados da validação cruzada GroupKFold(n=5) por song_id.',
        '> O objetivo não foi provar que a fusão melhora, mas medir empiricamente se há ganho real e sob quais condições.',
        '',
    ]

    for target in targets_available:
        linhas.append(f'## {target.capitalize()}')

        # cenários de fusão
        fusion_targets = scenario_summary[scenario_summary['target'] == target] if not scenario_summary.empty else pd.DataFrame()

        if fusion_targets.empty:
            linhas.append('_Sem dados de fusão disponíveis._')
            continue

        # fingerprint melhora openSMILE?
        ganhos = fusion_targets['ganho'].value_counts()
        n_consistente = ganhos.get('melhora consistente', 0)
        n_erro = ganhos.get('melhora em erro', 0)
        n_expl = ganhos.get('melhora em explicação', 0)
        n_sem = ganhos.get('sem melhora relevante', 0)
        total_cen = len(fusion_targets)

        linhas.append(f'**O fingerprint melhora o openSMILE em {target}?**')
        if n_consistente > 0:
            linhas.append(f'- Sim, com melhora consistente em {n_consistente}/{total_cen} cenários.')
        elif n_erro + n_expl > 0:
            linhas.append(f'- Melhora parcial: {n_erro} cenários com redução de erro, {n_expl} com aumento de R²/Pearson.')
        else:
            linhas.append(f'- Não houve melhora relevante nos {total_cen} cenários testados.')

        # melhor cenário de fusão
        best_row = fusion_targets.sort_values('delta_rmse').iloc[0]
        linhas.append(f'- Melhor cenário de fusão: **{scenario_label(best_row["scenario"])}** '
                      f'(Δ RMSE = {best_row["delta_rmse"]:+.4f}, '
                      f'Δ R² = {best_row["delta_r2"]:+.4f}, '
                      f'Δ Pearson = {best_row["delta_pearson"]:+.4f})')

        # qual tipo de fingerprint ajuda mais?
        fp_only = fusion_targets[fusion_targets['scenario'].isin(['2_fprank_only','3_fpband_only'])]
        if len(fp_only) >= 2:
            fp_rank_rmse = fp_only[fp_only['scenario']=='2_fprank_only']['delta_rmse'].values
            fp_band_rmse = fp_only[fp_only['scenario']=='3_fpband_only']['delta_rmse'].values
            if len(fp_rank_rmse) and len(fp_band_rmse):
                if fp_rank_rmse[0] < fp_band_rmse[0]:
                    linhas.append('- **Fingerprint Rank** apresenta melhor desempenho isolado do que Fingerprint Band.')
                elif fp_band_rmse[0] < fp_rank_rmse[0]:
                    linhas.append('- **Fingerprint Band** apresenta melhor desempenho isolado do que Fingerprint Rank.')
                else:
                    linhas.append('- Fingerprint Rank e Band têm desempenho similar isolados.')

        # consistência
        linhas.append(f'**A fusão melhora de forma consistente?**')
        if n_consistente >= 2:
            linhas.append(f'- Sim: {n_consistente} cenários com melhora consistente (erro + explicação).')
        elif n_erro + n_expl >= 2:
            linhas.append('- Parcialmente: a melhora não é consistente em todas as dimensões avaliadas.')
        else:
            linhas.append('- Não: os fingerprints não proporcionam ganho sistemático sobre o openSMILE.')

        # importância
        if not imp_by_origin.empty:
            sub_imp = imp_by_origin[
                (imp_by_origin['target'] == target) &
                (imp_by_origin['scenario'] == '7_os_allfingerprint')
            ]
            if not sub_imp.empty:
                grp = sub_imp.groupby('origin')['importance_pct'].mean().sort_values(ascending=False)
                linhas.append('**As features de fingerprint aparecem entre as mais importantes?**')
                for origin, pct in grp.items():
                    linhas.append(f'  - {origin}: {pct:.1f}% da importância total')
                fp_pct = grp.drop('openSMILE', errors='ignore').sum()
                if fp_pct > 30:
                    linhas.append(f'- Sim: fingerprints respondem por {fp_pct:.1f}% da importância total no cenário OS+AllFP.')
                else:
                    linhas.append(f'- Os fingerprints respondem por {fp_pct:.1f}% — openSMILE ainda domina a importância.')

        # complementaridade vs redundância
        linhas.append('**O resultado sugere complementaridade ou redundância?**')
        os_row = fusion_targets[fusion_targets['scenario']=='7_os_allfingerprint']
        if not os_row.empty:
            d = os_row.iloc[0]['delta_rmse']
            if d < -0.005:
                linhas.append('- **Complementaridade**: a fusão reduz o erro em relação ao openSMILE isolado, indicando que os fingerprints trazem informação adicional.')
            elif d > 0.005:
                linhas.append('- **Redundância ou ruído**: a fusão piora o desempenho, sugerindo que os fingerprints introduzem ruído ou são redundantes com o openSMILE.')
            else:
                linhas.append('- **Resultado neutro**: a fusão não altera significativamente o desempenho; os fingerprints podem ser parcialmente redundantes com o openSMILE.')

        linhas.append('')

    linhas += [
        '---',
        f'_Gerado em: {pd.Timestamp.now().strftime("%Y-%m-%d %H:%M")} | Validação: GroupKFold(n=5) por song_id_',
    ]
    return '\n'.join(linhas)


synthesis_text = sintetizar(delta_df, scenario_summary, imp_by_origin, targets_available, cfg)
display(Markdown(synthesis_text))

with open(cfg.output_dir() / 'fusion_interpretation_summary.md', 'w', encoding='utf-8') as f:
    f.write(synthesis_text)
print('Síntese exportada:', cfg.output_dir() / 'fusion_interpretation_summary.md')

# Síntese: Fusão openSMILE + Fingerprints Acústicos

> Este resumo foi gerado automaticamente a partir dos resultados da validação cruzada GroupKFold(n=5) por song_id.
> O objetivo não foi provar que a fusão melhora, mas medir empiricamente se há ganho real e sob quais condições.

## Valence
**O fingerprint melhora o openSMILE em valence?**
- Melhora parcial: 0 cenários com redução de erro, 1 com aumento de R²/Pearson.
- Melhor cenário de fusão: **6. OS+FPBand** (Δ RMSE = -0.0017, Δ R² = +0.0113, Δ Pearson = +0.0125)
- **Fingerprint Rank** apresenta melhor desempenho isolado do que Fingerprint Band.
**A fusão melhora de forma consistente?**
- Não: os fingerprints não proporcionam ganho sistemático sobre o openSMILE.
**As features de fingerprint aparecem entre as mais importantes?**
  - openSMILE: 66.3% da importância total
  - Fingerprint Band: 21.1% da importância total
  - Fingerprint Rank: 12.7% da importância total
- Sim: fingerprints respondem por 33.7% da importância total no cenário OS+AllFP.
**O resultado sugere complementaridade ou redundância?**
- **Resultado neutro**: a fusão não altera significativamente o desempenho; os fingerprints podem ser parcialmente redundantes com o openSMILE.

## Arousal
**O fingerprint melhora o openSMILE em arousal?**
- Sim, com melhora consistente em 3/7 cenários.
- Melhor cenário de fusão: **7. OS+AllFP** (Δ RMSE = -0.0075, Δ R² = +0.0341, Δ Pearson = +0.0223)
- **Fingerprint Band** apresenta melhor desempenho isolado do que Fingerprint Rank.
**A fusão melhora de forma consistente?**
- Sim: 3 cenários com melhora consistente (erro + explicação).
**As features de fingerprint aparecem entre as mais importantes?**
  - openSMILE: 45.1% da importância total
  - Fingerprint Band: 40.7% da importância total
  - Fingerprint Rank: 14.2% da importância total
- Sim: fingerprints respondem por 54.9% da importância total no cenário OS+AllFP.
**O resultado sugere complementaridade ou redundância?**
- **Complementaridade**: a fusão reduz o erro em relação ao openSMILE isolado, indicando que os fingerprints trazem informação adicional.

---
_Gerado em: 2026-06-08 04:53 | Validação: GroupKFold(n=5) por song_id_

Síntese exportada: C:\dev\python\TCC Ajustado\Dados\pycaret_fusion_validation_regression\fusion_interpretation_summary.md
